# Шаг 08 — подтверждение опасных действий

Новые инструменты `write_file` и `run_command` меняют мир. Наш узел `tools` перед ними вызывает `interrupt(...)`: граф **встаёт на паузу** (состояние в SQLite) и ждёт человека. Продолжаем через `Command(resume="y")` — с той же строки.

In [ ]:
import os

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()  # reads OPENROUTER_API_KEY and MODEL from .env

llm = ChatOpenAI(
    model=os.getenv("MODEL", "openai/gpt-4o-mini"),
    base_url="https://openrouter.ai/api/v1",   # OpenRouter speaks the OpenAI protocol
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

In [ ]:
from tools import DANGEROUS, list_files, read_file, run_command, write_file

TOOLS = [list_files, read_file, write_file, run_command]

SYSTEM = ("Ты ассистент по коду. Смотри реальные файлы через инструменты, "
          "ничего не выдумывай. Можешь создавать файлы и запускать команды. "
          "Если пользователь отказал в действии — не повторяй его, а спроси, "
          "что сделать вместо этого. Отвечай кратко.")

Агент будет создавать файлы — пусть работает в отдельной папке `playground/`. Ячейку можно перезапускать: каждый раз песочница чистая.

In [ ]:
import shutil
from pathlib import Path

if Path.cwd().name == "playground":       # re-run safe
    os.chdir("..")
shutil.rmtree("playground", ignore_errors=True)   # fresh sandbox for the demo
os.makedirs("playground")
os.chdir("playground")
print("working in:", os.getcwd())

## Узел `tools` с паузой

In [ ]:
from langchain_core.messages import ToolMessage
from langgraph.graph import START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import Command, interrupt

llm_with_tools = llm.bind_tools(TOOLS)
run_tools = ToolNode(TOOLS)


def agent(state: MessagesState) -> dict:
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


def tools(state: MessagesState) -> dict:
    """Run the requested tools — but ask a human first for dangerous ones."""
    calls = state["messages"][-1].tool_calls
    risky = [c for c in calls if c["name"] in DANGEROUS]
    if risky:
        answer = interrupt(risky)                  # ← the graph pauses here
        if answer != "y":
            return {"messages": [
                ToolMessage("The user DENIED this action. Do not retry it — "
                            "tell the user and ask what to do instead.",
                            name=c["name"], tool_call_id=c["id"])
                for c in calls
            ]}
    return run_tools.invoke(state)

In [ ]:
import sqlite3

from langgraph.checkpoint.sqlite import SqliteSaver

builder = StateGraph(MessagesState)
builder.add_node("agent", agent)
builder.add_node("tools", tools)                   # our node instead of ToolNode
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)
builder.add_edge("tools", "agent")

# interrupt needs a checkpointer: the paused state has to live somewhere
graph = builder.compile(checkpointer=SqliteSaver(sqlite3.connect("memory.sqlite", check_same_thread=False)))

Помощники: `ask` — новый вопрос, `resume` — ответ на паузу.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

config = {"configurable": {"thread_id": "demo"}}


def run(graph_input) -> None:
    for update in graph.stream(graph_input, config, stream_mode="updates"):
        if "__interrupt__" in update:
            for call in update["__interrupt__"][0].value:
                print(f"  ⏸ ждёт подтверждения: {call['name']}({call['args']})")
            continue
        for node, data in update.items():
            for msg in data["messages"]:
                if node == "agent" and msg.tool_calls:
                    for call in msg.tool_calls:
                        print(f"  [agent] → {call['name']}({call['args']})")
                elif node == "tools":
                    status = "denied" if msg.content.startswith("The user DENIED") else "done"
                    print(f"  [tools] {msg.name} {status}")
                else:
                    print("ai ›", msg.content)


def ask(text: str) -> None:
    new = [HumanMessage(text)]
    if not graph.get_state(config).values:
        new.insert(0, SystemMessage(SYSTEM))
    run({"messages": new})


def resume(answer: str) -> None:
    if not graph.get_state(config).next:
        print("нечего подтверждать — граф не на паузе")
        return
    run(Command(resume=answer))

## Просим создать и запустить файл

In [ ]:
ask("Создай hello.py, который печатает слово Hello, и запусти его.")

Граф **стоит на паузе**. Состояние сохранено — можно посмотреть, где он остановился и чего ждёт:

In [ ]:
state = graph.get_state(config)
print("next node:", state.next)
print("waiting for:", [(c["name"], c["args"]) for c in state.interrupts[0].value])

In [ ]:
resume("y")

Если модель попросила ещё одно опасное действие — снова пауза. Подтверждаем:

In [ ]:
resume("y")

In [ ]:
print(open("hello.py").read())

## Отказ

Ответ «n» — тоже ответ: модель получает «user denied» и меняет план.

In [ ]:
ask("Удали hello.py командой rm.")

In [ ]:
resume("n")

In [ ]:
os.path.exists("hello.py")   # the file is still here

---
# Посложнее

Дальше два более сложных сценария. Сначала общий помощник для печати: он показывает шаги узлов (в том числе **внутри подграфов** — с отступом) и паузы.

In [ ]:
def show_pause(value) -> None:
    if isinstance(value, dict) and "plan" in value:
        print("⏸ план ждёт одобрения:")
        for i, step in enumerate(value["plan"], 1):
            print(f"   {i}. {step}")
    else:
        for call in value:
            print(f"⏸ ждёт решения: {call['name']}({call['args']})")


def show_update(node: str, data: dict, indent: str = "") -> None:
    if node == "planner":
        for i, step in enumerate(data["plan"], 1):
            print(f"{indent}[planner] {i}. {step}")
    elif node == "review_plan":
        print(f"{indent}[review_plan] {'✅ план принят' if data['approved'] else '❌ план отклонён'}")
    else:
        for msg in data.get("messages", []):
            if msg.type == "ai" and msg.tool_calls:
                label = "вызов исправлен человеком:" if node == "tools" else "→"
                for call in msg.tool_calls:
                    print(f"{indent}[{node}] {label} {call['name']}({call['args']})")
            elif msg.type == "tool":
                first_line = str(msg.content).splitlines()[0] if msg.content else ""
                print(f"{indent}[tools] {msg.name}: {first_line[:80]}")
            elif msg.type == "human":
                print(f"{indent}[{node}] 👤 {msg.content}")
            elif msg.type == "ai":
                print(f"{indent}[{node}] {msg.content}")


def run_on(g, cfg: dict, graph_input, skip: tuple = ()) -> None:
    """Stream a run: print every node step (also inside subgraphs) and pauses."""
    for namespace, update in g.stream(graph_input, cfg, stream_mode="updates", subgraphs=True):
        indent = "      " if namespace else ""
        for node, data in update.items():
            if node == "__interrupt__":
                show_pause(data[0].value)
            elif node not in skip and data:
                show_update(node, data, indent)

## A. Ревью вызова: принять / исправить / ответить

`interrupt` может вернуть **любые данные**, не только «y/n». Дадим человеку три действия, как в Claude Code:

| решение | что происходит |
|---|---|
| `accept` | вызов выполняется как есть |
| `edit` | человек меняет аргументы; исправленное сообщение модели **заменяет** исходное в истории (тот же `id` → `add_messages` не добавляет, а заменяет) — как будто модель так и попросила |
| `feedback` | вызов отменяется, а текст человека добавляется **новым сообщением пользователя** — модель перестраивает план |

`parallel_tool_calls=False` — модель делает по одному действию за раз, так проще ревьюить.

In [ ]:
llm_one_at_a_time = llm.bind_tools(TOOLS, parallel_tool_calls=False)


def agent_one(state: MessagesState) -> dict:
    return {"messages": [llm_one_at_a_time.invoke(state["messages"])]}


def tools_with_review(state: MessagesState) -> dict:
    """Before a dangerous call let a human accept, edit or answer with feedback."""
    ai = state["messages"][-1]
    risky = [c for c in ai.tool_calls if c["name"] in DANGEROUS]
    if not risky:
        return run_tools.invoke(state)

    decision = interrupt(risky)          # e.g. {"action": "edit", "args": {...}}

    if decision["action"] == "feedback":
        cancelled = [ToolMessage("Cancelled by the user.", name=c["name"], tool_call_id=c["id"])
                     for c in ai.tool_calls]
        return {"messages": [*cancelled, HumanMessage(decision["text"])]}   # the human speaks
    if decision["action"] == "edit":
        target = risky[0]["id"]
        calls = [{**c, "args": decision["args"]} if c["id"] == target else c for c in ai.tool_calls]
        ai = ai.model_copy(update={"tool_calls": calls})   # same id → REPLACES the message
        return {"messages": [ai, *run_tools.invoke({"messages": [ai]})["messages"]]}
    return run_tools.invoke(state)        # "accept"


review_builder = StateGraph(MessagesState)
review_builder.add_node("agent", agent_one)
review_builder.add_node("tools", tools_with_review)
review_builder.add_edge(START, "agent")
review_builder.add_conditional_edges("agent", tools_condition)
review_builder.add_edge("tools", "agent")
review_graph = review_builder.compile(checkpointer=graph.checkpointer)   # same SQLite
review_config = {"configurable": {"thread_id": "review"}}


def ask_review(text: str) -> None:
    new = [HumanMessage(text)]
    if not review_graph.get_state(review_config).values:
        new.insert(0, SystemMessage(SYSTEM))
    run_on(review_graph, review_config, {"messages": new})


def decide(**decision) -> None:
    """decide(action="accept") | decide(action="edit", args={...}) | decide(action="feedback", text="...")"""
    if not review_graph.get_state(review_config).next:
        print("нечего решать — граф не на паузе")
        return
    run_on(review_graph, review_config, Command(resume=decision))

### Исправляем вызов

In [ ]:
ask_review("Создай файл notes.txt со строкой hello, затем покажи его содержимое командой cat.")

Модель хочет записать `hello`. Исправим содержимое прямо в вызове:

In [ ]:
decide(action="edit", args={"path": "notes.txt", "content": "hello from a human\n"})

Теперь она просит `cat` — принимаем как есть. В выводе должен быть **наш** текст:

In [ ]:
decide(action="accept")

В истории остался исправленный вызов — модель «видит», будто сама так и попросила:

In [ ]:
for message in review_graph.get_state(review_config).values["messages"]:
    for call in getattr(message, "tool_calls", None) or []:
        if call["name"] == "write_file":
            print(call["args"])

### Отвечаем вместо выполнения

In [ ]:
ask_review("Удали notes.txt.")

In [ ]:
decide(action="feedback", text="Не удаляй — переименуй в notes.bak")

Модель перестроила план. Если она предлагает переименование — принимаем:

In [ ]:
decide(action="accept")

In [ ]:
sorted(os.listdir("."))

## B. План → одобрение → исполнение

Одобрять каждое действие утомительно. Другой подход — человек одобряет **план целиком**:

```
START → planner → review_plan ─(одобрен)─→ executor → END
                       └──(отклонён)──→ END
```

- `planner` — structured output (шаг 03) возвращает список шагов;
- `review_plan` — пауза: человек принимает, правит или отклоняет план;
- `executor` — агент с инструментами (`create_agent`, шаг 06) **как подграф**. Он выполняет план без вопросов по каждому действию: одобрение плана и есть разрешение.

Состояние графа — уже не только `messages`: ещё `task`, `plan`, `approved`.

In [ ]:
from langchain.agents import create_agent
from langgraph.graph import END
from pydantic import BaseModel, Field


class PlanState(MessagesState):
    task: str
    plan: list[str]
    approved: bool


class Plan(BaseModel):
    """A short step-by-step plan."""

    steps: list[str] = Field(description="2-4 concrete actions with files or shell commands")


planner_llm = llm.with_structured_output(Plan, method="function_calling")


def planner(state: PlanState) -> dict:
    plan = planner_llm.invoke([SystemMessage("Составь короткий план выполнения задачи. Пиши по-русски."),
                               HumanMessage(state["task"])])
    return {"plan": plan.steps}


def review_plan(state: PlanState) -> dict:
    decision = interrupt({"plan": state["plan"]})   # {"action": "approve" | "edit" | "reject", ...}
    if decision["action"] == "reject":
        return {"approved": False}
    plan = decision.get("plan", state["plan"])       # "edit" brings a new plan
    steps = "\n".join(f"{i}. {s}" for i, s in enumerate(plan, 1))
    return {"plan": plan, "approved": True,
            "messages": [HumanMessage(f"Задача: {state['task']}\nВыполни план по шагам:\n{steps}")]}


executor = create_agent(llm, TOOLS, system_prompt="Ты исполнитель. Выполни план с помощью "
                        "инструментов и кратко отчитайся по-русски.")

plan_builder = StateGraph(PlanState)
plan_builder.add_node("planner", planner)
plan_builder.add_node("review_plan", review_plan)
plan_builder.add_node("executor", executor)        # an agent as a subgraph
plan_builder.add_edge(START, "planner")
plan_builder.add_edge("planner", "review_plan")
plan_builder.add_conditional_edges("review_plan", lambda s: "executor" if s["approved"] else END,
                                   {"executor": "executor", END: END})
plan_builder.add_edge("executor", END)
plan_graph = plan_builder.compile(checkpointer=graph.checkpointer)

Схема (`xray=True` раскрывает исполнителя — внутри тот же цикл model ⇄ tools; картинку рисует mermaid.ink, без интернета будет ASCII):

In [ ]:
from IPython.display import Image, display

try:
    display(Image(plan_graph.get_graph(xray=True).draw_mermaid_png()))
except Exception:
    print(plan_graph.get_graph(xray=True).draw_ascii())

In [ ]:
import uuid


def start_plan(task: str) -> None:
    """Every task gets its own thread."""
    global plan_config
    plan_config = {"configurable": {"thread_id": f"plan-{uuid.uuid4().hex[:6]}"}}
    run_on(plan_graph, plan_config, {"task": task})


def decide_plan(**decision) -> None:
    """decide_plan(action="approve") | decide_plan(action="edit", plan=[...]) | decide_plan(action="reject")"""
    run_on(plan_graph, plan_config, Command(resume=decision), skip=("executor",))

### Правим план и запускаем

In [ ]:
start_plan("Создай fizzbuzz.py для чисел от 1 до 15, запусти его и сохрани вывод в fizzbuzz_out.txt.")

Граф на паузе, план — в состоянии. Добавим свой шаг в конец и одобрим:

In [ ]:
plan = plan_graph.get_state(plan_config).values["plan"]
decide_plan(action="edit", plan=plan + ["Покажи содержимое fizzbuzz_out.txt командой cat"])

In [ ]:
print(open("fizzbuzz_out.txt").read())

### Отклоняем опасный план

При отказе граф уходит в `END`, исполнитель даже не запускается:

In [ ]:
start_plan("Удали все файлы в текущей папке.")

In [ ]:
decide_plan(action="reject")
print("файлы на месте:", sorted(os.listdir(".")))

**Итог.** Два стиля контроля: **ревью каждого действия** (точно, но утомительно) и **одобрение плана** (быстро, но доверяем исполнителю на всём плане). В реальных агентах их комбинируют: план одобряется целиком, а самые опасные действия всё равно спрашивают отдельно.